<a href="https://colab.research.google.com/github/sevaradevoloper/machine-learning-class-work/blob/main/netflixtv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [128]:
# CatBoost

In [129]:
# Dataset tanlash va yuklab olish

In [130]:
!pip install opendatasets legacy-cgi --upgrade --quiet


In [131]:
!pip install catboost

In [132]:
import opendatasets as od

In [133]:
od.download("https://www.kaggle.com/datasets/shivamb/netflix-shows")

Skipping, found downloaded files in "./netflix-shows" (use force=True to force download)


In [134]:
import pandas as pd




In [135]:
df = pd.read_csv("/content/netflix-shows/netflix_titles.csv")

In [136]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


show_id — ID raqami (kino tartib raqami).

type — Turi (Kino yoki Serial).

title — Nomi (kino yoki serial nomi).

director — Rejissyor (film ijodkori).

cast — Aktyorlar (bosh roldagilar).

country — Davlat (kino olingan mamlakat).

date_added — Qo'shilgan sana (Netflix'ga yuklangan kun).

release_year — Chiqqan yili (ekranga chiqqan yili).

rating — Yosh cheklovi (kimlar ko'rishi mumkinligi).

duration — Davomiyligi (vaqti: daqiqa yoki fasl).

listed_in — Janri (kino qaysi turga kirishi).

description — Qisqacha tavsif (kino mazmuni).

In [137]:
print(df.shape)

(8807, 12)


In [138]:
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [139]:
df["type"].value_counts()

# imbalanced target

,count
type,
Movie,6131
TV Show,2676


In [140]:
# EDA (datasetni tanish) — qanday ustunlar bor, qaysilari kategorik

duration — Movie uchun "90 min", TV Show uchun "2 Seasons" deb yozilgan. Bu to'g'ridan-to'g'ri type'ni aytib beradi! → Bu ustunni tashlab yuborishimiz kerak (yoki keyinroq alohida ehtiyotkorlik bilan ishlash).

show_id, title, description, cast, director — bular deyarli har qatorda noyob (unique) qiymatlar. Ular model uchun foyda bermaydi va CatBoost'ni chalg'itadi. Bularni ham tashlaймiz.


Qoladigan foydali ustunlar (feature'lar):

country (kategorik) ✅
rating (kategorik — TV-MA, PG va h.k.) ✅
listed_in (kategorik — janrlar) ✅
release_year (raqamli) ✅
date_added'dan yil/oy ajratib olsa bo'ladi (keyinroq, ixtiyoriy)

In [141]:
drop_col = ['show_id',
 'title',
 'director',
 'cast',
 'duration',
 'description']

In [142]:
df = df.drop(columns=df[drop_col])
df

,type,country,date_added,release_year,rating,listed_in
0,Movie,United States,"September 25, 2021",2020,PG-13,Documentaries
1,TV Show,South Africa,"September 24, 2021",2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,TV Show,NaN,"September 24, 2021",2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,TV Show,NaN,"September 24, 2021",2021,TV-MA,"Docuseries, Reality TV"
4,TV Show,India,"September 24, 2021",2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...,...
8802,Movie,United States,"November 20, 2019",2007,R,"Cult Movies, Dramas, Thrillers"
8803,TV Show,NaN,"July 1, 2019",2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,Movie,United States,"November 1, 2019",2009,R,"Comedies, Horror Movies"
8805,Movie,United States,"January 11, 2020",2006,PG,"Children & Family Movies, Comedies"


In [143]:
# isnull
df.isnull().sum()

,0
type,0
country,831
date_added,10
release_year,0
rating,4
listed_in,0


In [144]:
cat_cl = ['date_added','country','rating']
cat_mode = df[cat_cl].mode().iloc[0]
df[cat_cl] = df[cat_cl].fillna(cat_mode)
df.isnull().sum()

,0
type,0
country,0
date_added,0
release_year,0
rating,0
listed_in,0


In [145]:
df = df.drop(columns=['date_added'])
df

,type,country,release_year,rating,listed_in
0,Movie,United States,2020,PG-13,Documentaries
1,TV Show,South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,TV Show,United States,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,TV Show,United States,2021,TV-MA,"Docuseries, Reality TV"
4,TV Show,India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...
8802,Movie,United States,2007,R,"Cult Movies, Dramas, Thrillers"
8803,TV Show,United States,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,Movie,United States,2009,R,"Comedies, Horror Movies"
8805,Movie,United States,2006,PG,"Children & Family Movies, Comedies"


In [146]:
# X, y ajratish + kategorik ustunlarni belgilash + train/test split

df["type"] = df["type"].replace({'Movie': 1,'TV Show': 0})
df

/tmp/ipykernel_881/3342547742.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["type"] = df["type"].replace({'Movie': 1,'TV Show': 0})


,type,country,release_year,rating,listed_in
0,1,United States,2020,PG-13,Documentaries
1,0,South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,0,United States,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,0,United States,2021,TV-MA,"Docuseries, Reality TV"
4,0,India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...
8802,1,United States,2007,R,"Cult Movies, Dramas, Thrillers"
8803,0,United States,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,1,United States,2009,R,"Comedies, Horror Movies"
8805,1,United States,2006,PG,"Children & Family Movies, Comedies"


In [147]:
# X va y
X = df.drop(columns=['type'])
y = df['type']

In [148]:
# cat_features
cat_features = X.select_dtypes(include='object').columns.tolist()
cat_features

['country', 'rating', 'listed_in']

In [149]:
# train_test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state = 42,stratify= y)
print(f'X_train shape {X_train.shape}')
print(f'X_test shape:{X_test.shape}')
print(f'y_train shape {y_train.shape}')
print(f'y_test shape:{y_test.shape}')

X_train shape (7045, 4)
X_test shape:(1762, 4)
y_train shape (7045,)
y_test shape:(1762,)


In [150]:
# CatBoost bilan train
from catboost import CatBoostClassifier

In [151]:
import time

In [152]:
start = time.perf_counter()
model = CatBoostClassifier(
    iterations= 300,
    learning_rate = 0.1,
    depth = 6,
    random_state=42,
    verbose = 100
)

model.fit(X_train,y_train,cat_features = cat_features)

0:	learn: 0.4060501	total: 25.1ms	remaining: 7.5s
100:	learn: 0.0258537	total: 1.59s	remaining: 3.13s
200:	learn: 0.0230836	total: 3.15s	remaining: 1.55s
299:	learn: 0.0209859	total: 5.16s	remaining: 0us


CatBoostClassifier(depth=6, iterations=300, learning_rate=0.1, random_state=42, verbose=100)

In [153]:
catboost_time = time.perf_counter() - start
print(f'Train uchun olingan vaqt : {catboost_time}')

Train uchun olingan vaqt : 5.339855057998648


In [154]:
# predict
y_pred = model.predict(X_test)

In [155]:
from sklearn.metrics import accuracy_score


In [156]:
# baholash

acc = accuracy_score(y_test,y_pred)
print(f'Accurancy score {acc}')

Accurancy score 0.9977298524404086


In [157]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,listed_in,60.615810
1,rating,17.060122
2,country,12.189776
3,release_year,10.134292


In [ ]:
from sklearn.preprocessing import OrdinalEncoder